# Retrain SATD Detector (Yikun Li et al., EMSE 2023)
Notebook ini khusus untuk **melatih ulang (*retrain*) arsitektur asli Multitask Text-CNN** dari Yikun Li sebagaimana didefinisikan pada script resmi `satd_detector.py` ([yikun-li/satd-different-sources-data](https://github.com/yikun-li/satd-different-sources-data)).

### Fitur Utama Notebook Ini:
1. **Fokus Langsung ke Model Utama (No Baseline):** Langsung melatih model `TextCNNMultitask` (kernel `[1, 2, 3, 4, 5]`, 200 filter, 4 task heads) tanpa membuang waktu menjalankan Random, LR, SVM, dan RF.
2. **Auto-Download Dataset:** Otomatis mengunduh 4 file CSV dari repositori GitHub Yikun Li saat dijalankan di Kaggle atau Colab.
3. **Optimal untuk Kaggle (GPU T4 x2 / T4):** Memanfaatkan akselerasi PyTorch CUDA.
4. **Format Model Kompatibel:** Menyimpan bobot model ke file `satd_detector.pt` yang kompatibel dengan script `satd_detector.py` milik penulis.
5. **Inferensi Interaktif:** Dilengkapi fungsi pengujian langsung (`classify_satd`) untuk menguji kalimat custom atau contoh bawaan penulis (`simple_test`).

## Cell 1: Environment & GPU Setup

In [ ]:
import os
import re
import json
import string
import random
import warnings
import numpy as np
import pandas as pd
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import precision_recall_fscore_support, classification_report

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

warnings.filterwarnings('ignore')

# Seed Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Deteksi Akselerator Hardware (Kaggle T4 x2 / Colab GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(i) for i in range(num_gpus)] if num_gpus > 0 else []
print(f'Device: {device} | Total GPU: {num_gpus} {gpu_names}')

# Hyperparameter Pelatihan (Bisa disesuaikan)
EPOCHS = 10         # Jumlah epoch (10 epoch standar konvergensi)
BATCH_SIZE = 64     # Ukuran mini-batch per step
LEARNING_RATE = 1e-3
EMBED_DIM = 300     # Dimensi representasi kata
KERNEL_NUM = 200    # 200 filter per ukuran kernel (sesuai satd_detector.py)
KERNEL_SIZES = [1, 2, 3, 4, 5]  # Ukuran kernel n-gram (sesuai satd_detector.py)
DROPOUT = 0.5

## Cell 2: Auto-Download Dataset dari GitHub Repo Yikun Li
Mengambil 4 dataset CSV dari repositori `https://github.com/yikun-li/satd-different-sources-data` ke folder `./data/`.

In [ ]:
GITHUB_REPO_URL = 'https://github.com/yikun-li/satd-different-sources-data.git'
RAW_BASE_URL = 'https://raw.githubusercontent.com/yikun-li/satd-different-sources-data/main'
DATA_DIR = './data'

required_files = [
    'satd-dataset-code_comments.csv',
    'satd-dataset-commit_messages.csv',
    'satd-dataset-pull_requests.csv',
    'satd-dataset-issues.csv'
]

os.makedirs(DATA_DIR, exist_ok=True)
all_present = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in required_files)

if not all_present:
    print('Mengunduh dataset dari repositori GitHub Yikun Li...')
    exit_code = os.system(f'git clone {GITHUB_REPO_URL} /tmp/satd_repo && cp /tmp/satd_repo/*.csv {DATA_DIR}/ && rm -rf /tmp/satd_repo')
    if exit_code != 0 or not all(os.path.exists(os.path.join(DATA_DIR, f)) for f in required_files):
        for f in required_files:
            dest = os.path.join(DATA_DIR, f)
            if not os.path.exists(dest):
                url = f'{RAW_BASE_URL}/{f}'
                print(f'Mengunduh {f}...')
                urllib.request.urlretrieve(url, dest)
    print('Seluruh file dataset berhasil diunduh ke folder:', DATA_DIR)
else:
    print('Dataset sudah lengkap di folder:', DATA_DIR)

## Cell 3: Preprocessing & Label Mapping Sesuai `satd_detector.py`
Menerapkan konstanta pemetaan task dan label persis dari `satd_detector.py` milik penulis:
* `DEF_MAPPING`: `{'issue': 0, 'commit_message': 1, 'code_comment': 2, 'pull_request': 3}`
* `DEF_LABELS`: `['non-SATD', 'code|design-debt', 'requirement-debt', 'documentation-debt', 'test-debt']`

In [ ]:
# Definisi Konstanta Persis dari satd_detector.py
DEF_ISSUE = 'issue'
DEF_COMMIT = 'commit_message'
DEF_COMMENT = 'code_comment'
DEF_PULL = 'pull_request'

DEF_MAPPING = {
    DEF_ISSUE: 0,
    DEF_COMMIT: 1,
    DEF_COMMENT: 2,
    DEF_PULL: 3
}

DEF_LABELS = [
    'non-SATD',            # Index 0
    'code|design-debt',    # Index 1
    'requirement-debt',    # Index 2
    'documentation-debt',  # Index 3
    'test-debt'            # Index 4
]

DATA_FILES = {
    DEF_COMMENT: os.path.join(DATA_DIR, 'satd-dataset-code_comments.csv'),
    DEF_COMMIT: os.path.join(DATA_DIR, 'satd-dataset-commit_messages.csv'),
    DEF_PULL: os.path.join(DATA_DIR, 'satd-dataset-pull_requests.csv'),
    DEF_ISSUE: os.path.join(DATA_DIR, 'satd-dataset-issues.csv')
}

def map_raw_label(raw_label):
    raw = str(raw_label).lower().strip()
    if 'code' in raw or 'design' in raw:
        return 1  # code|design-debt
    elif 'requirement' in raw:
        return 2  # requirement-debt
    elif 'documentation' in raw:
        return 3  # documentation-debt
    elif 'test' in raw:
        return 4  # test-debt
    else:
        return 0  # non-SATD

# Fungsi Preprocessing Kata Persis dari satd_detector.py
tokenizer_words = nltk.TweetTokenizer()
punctuation_chars = string.punctuation.replace('!', '').replace('?', '')
padding_token = '<pad>'

def comment_pre_processing(comment):
    if not isinstance(comment, str) or not comment.strip():
        return ['<pad>'] * 5
    comment = re.sub(r'(//)|(/\*)|(\*/)', '', comment).lower()
    try:
        tokens_sentences = [tokenizer_words.tokenize(t) for t in nltk.sent_tokenize(comment)]
    except Exception:
        tokens_sentences = [tokenizer_words.tokenize(comment)]

    processed_tokens_sentences = []
    for sentence in tokens_sentences:
        processed_tokens = []
        for token in sentence:
            if token == 'non-nls' or token == padding_token:
                processed_tokens.append(token)
                continue
            elif token == ',':
                processed_tokens.append('.')
            elif any(c in token for c in [' ', '.', '#', '_', '/', '>', '<', '@']):
                continue
            elif any(c.isdigit() for c in token):
                continue
            else:
                processed_tokens.append(token)

        if len(processed_tokens) > 0 and processed_tokens[-1] != '.':
            processed_tokens.append('.')
        processed_tokens_sentences.append(processed_tokens)

    tokens = [w for t in processed_tokens_sentences for w in t]
    stripped = [w for w in tokens if w and (w not in punctuation_chars and ':' not in w and '=' not in w and ')' not in w and '(' not in w)]
    if len(stripped) < 5:
        stripped = stripped + [padding_token] * (5 - len(stripped))
    return stripped

# Load dan bersihkan data
dfs = {}
print('Memuat dan membersihkan seluruh dataset teks...')
for tp, file_path in DATA_FILES.items():
    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip')
    text_col = 'text' if 'text' in df.columns else df.columns[1]
    cls_col = 'classification' if 'classification' in df.columns else ('label' if 'label' in df.columns else df.columns[-1])
    
    df['clean_label'] = df[cls_col].apply(map_raw_label)
    df['tokens'] = df[text_col].astype(str).apply(comment_pre_processing)
    dfs[tp] = df[['tokens', 'clean_label']].copy()
    print(f'  {tp}: {len(df)} sampel dimuat.')

## Cell 4: Pembuatan Vocabulary & Dataset Loader

In [ ]:
class SATDVocabulary:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {'<pad>': 0, '<unk>': 1}
        self.idx2word = {0: '<pad>', 1: '<unk>'}
        self.counts = {}

    def build(self, token_lists):
        for toks in token_lists:
            for tok in toks:
                self.counts[tok] = self.counts.get(tok, 0) + 1
        idx = 2
        for tok, cnt in self.counts.items():
            if cnt >= self.min_freq and tok not in self.word2idx:
                self.word2idx[tok] = idx
                self.idx2word[idx] = tok
                idx += 1

    def numericalize(self, tokens, max_len=100):
        indices = [self.word2idx.get(t, 1) for t in tokens[:max_len]]
        if len(indices) < max_len:
            indices += [0] * (max_len - len(indices))
        return indices

    def __len__(self):
        return len(self.word2idx)

# Kumpulkan seluruh korpus token
all_tokens = []
for tp in dfs:
    all_tokens.extend(dfs[tp]['tokens'].tolist())

vocab = SATDVocabulary(min_freq=2)
vocab.build(all_tokens)
print(f'Total Vocabulary Tokens: {len(vocab)}')

# Split Train (90%) dan Validation (10%) Stratified per Task
train_data = {}
val_data = {}

for tp in dfs:
    X = dfs[tp]['tokens'].values
    y = dfs[tp]['clean_label'].values
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.10, random_state=42, stratify=y)
    train_data[tp] = (X_tr, y_tr)
    val_data[tp] = (X_va, y_va)
    print(f'  {tp}: Train = {len(X_tr)} | Val = {len(X_va)}')

class SATDTorchDataset(Dataset):
    def __init__(self, token_list, labels, vocab, max_len=100):
        self.features = [vocab.numericalize(toks, max_len) for toks in token_list]
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return torch.tensor(self.features[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_loaders = {}
val_loaders = {}

for tp in dfs:
    tr_ds = SATDTorchDataset(train_data[tp][0], train_data[tp][1], vocab)
    va_ds = SATDTorchDataset(val_data[tp][0], val_data[tp][1], vocab)
    train_loaders[tp] = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loaders[tp] = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False)

## Cell 5: Arsitektur Model Multitask Text-CNN (Identik dengan `satd_detector.py`)
Arsitektur ini memiliki:
* `list_conv`: Filter konvolusi ukuran window `[1, 2, 3, 4, 5]` dengan 200 filter (total 1000 feature maps).
* `dropout`: 0.5.
* `fcs`: 4 linear layer kepala output untuk 4 task, masing-masing memprediksi 5 kelas.
* `embed`: Embedding layer trainable (dimensi 300) yang dilatih *from scratch*.

In [ ]:
class TextCNNMultitaskTrainable(nn.Module):
    """
    Arsitektur TextCNN Multitask resmi Yikun Li (EMSE 2023) dengan trainable embedding.
    Struktur bobot identik dengan TextCNNMultitask pada satd_detector.py.
    """
    def __init__(self, vocab_size, embed_dim=EMBED_DIM, kernel_num=KERNEL_NUM, kernel_sizes=KERNEL_SIZES, dropout=DROPOUT):
        super(TextCNNMultitaskTrainable, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.list_conv = nn.ModuleList([
            nn.Conv2d(1, kernel_num, (size, embed_dim)) for size in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fcs = nn.ModuleList([
            nn.Linear(len(kernel_sizes) * kernel_num, len(DEF_LABELS)) for _ in range(len(DEF_MAPPING.keys()))
        ])

    def forward(self, x, task_id=None):
        # x shape: (batch_size, seq_len)
        x_embed = self.embed(x).unsqueeze(1)  # (batch_size, 1, seq_len, embed_dim)
        x_list_conv = [F.relu(conv(x_embed)).squeeze(3) for conv in self.list_conv]
        x_list_max_pool = [F.max_pool1d(x_c, x_c.size(2)).squeeze(2) for x_c in x_list_conv]
        x_cat = torch.cat(x_list_max_pool, 1)
        x_drop = self.dropout(x_cat)
        
        if task_id is not None:
            return self.fcs[task_id](x_drop)
        
        # Jika task_id tidak diberikan, return seluruh logit 4 task (kompatibel penuh dengan satd_detector.py)
        return [fc(x_drop) for fc in self.fcs]

model = TextCNNMultitaskTrainable(vocab_size=len(vocab)).to(device)
print(model)

## Cell 6: Stochastic Multitask Training Loop (Collobert & Weston 2008)
Melatih bersamaan ke-4 sumber data secara acak pada setiap langkah iterasi (*stochastic interleaving*).

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

source_tasks = [DEF_ISSUE, DEF_COMMIT, DEF_COMMENT, DEF_PULL]

print('=== MEMULAI RETRAINING MULTITASK SATD DETECTOR ===')
for epoch in range(1, EPOCHS + 1):
    model.train()
    iterators = {tp: iter(train_loaders[tp]) for tp in source_tasks}
    finished = {tp: False for tp in source_tasks}
    
    running_loss = {tp: 0.0 for tp in source_tasks}
    task_steps = {tp: 0 for tp in source_tasks}
    
    while not all(finished.values()):
        avail = [tp for tp in source_tasks if not finished[tp]]
        if not avail:
            break
        
        # Pilih task acak
        chosen_task = random.choice(avail)
        task_id = DEF_MAPPING[chosen_task]
        
        try:
            bx, by = next(iterators[chosen_task])
        except StopIteration:
            finished[chosen_task] = True
            continue
            
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx, task_id=task_id)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        
        running_loss[chosen_task] += loss.item()
        task_steps[chosen_task] += 1
        
    total_loss = sum(running_loss.values()) / max(sum(task_steps.values()), 1)
    print(f'Epoch [{epoch:02d}/{EPOCHS:02d}] - Average Loss: {total_loss:.4f}')
    for tp in source_tasks:
        avg_task_loss = running_loss[tp] / max(task_steps[tp], 1)
        print(f'    - {tp:<15} Loss: {avg_task_loss:.4f}')

print('\nPelatihan Selesai!')

## Cell 7: Evaluasi Model pada Data Validasi (F1-Score per Debt Type)

In [ ]:
model.eval()
eval_rows = []

print('=== EVALUASI MODEL PADA DATA VALIDASI ===\n')
with torch.no_grad():
    for tp in source_tasks:
        task_id = DEF_MAPPING[tp]
        preds, trues = [], []
        for bx, by in val_loaders[tp]:
            bx = bx.to(device)
            out = model(bx, task_id=task_id)
            pred = torch.argmax(out, dim=1).cpu().numpy()
            preds.extend(pred)
            trues.extend(by.numpy())
            
        # Metrik untuk 4 tipe technical debt: [1: C/D, 2: REQ, 3: DOC, 4: TST]
        p, r, f1, _ = precision_recall_fscore_support(trues, preds, labels=[1, 2, 3, 4], zero_division=0)
        avg_f1 = np.mean(f1)
        
        eval_rows.append({
            'Source Task': tp,
            'C/D. F1': f'{f1[0]:.3f}',
            'REQ. F1': f'{f1[1]:.3f}',
            'DOC. F1': f'{f1[2]:.3f}',
            'TST. F1': f'{f1[3]:.3f}',
            'AVG. F1 (Macro)': f'{avg_f1:.3f}'
        })

eval_df = pd.DataFrame(eval_rows)
print(eval_df.to_string(index=False))

## Cell 8: Simpan Bobot Model (`satd_detector.pt`) & Vocabulary
File bobot ini tersimpan dan dapat langsung dimuat ke objek `SATDDetector`.

In [ ]:
OUTPUT_MODEL_PATH = 'satd_detector.pt'
VOCAB_PATH = 'satd_vocab.json'

# Simpan Bobot Model PyTorch
torch.save(model.state_dict(), OUTPUT_MODEL_PATH)
print(f'Bobot model berhasil disimpan ke: {OUTPUT_MODEL_PATH}')

# Simpan Vocabulary Word2Idx
with open(VOCAB_PATH, 'w', encoding='utf-8') as f:
    json.dump(vocab.word2idx, f, ensure_ascii=False, indent=2)
print(f'Vocabulary berhasil disimpan ke: {VOCAB_PATH}')

## Cell 9: Pengujian Inferensi Langsung (Test Prediksi Kalimat)
Menjalankan suite pengujian identik dengan fungsi `simple_test` dari `satd_detector.py`.

In [ ]:
def classify_satd(text, source_type):
    model.eval()
    toks = comment_pre_processing(text)
    idx_seq = vocab.numericalize(toks, max_len=100)
    
    inp = torch.tensor([idx_seq], dtype=torch.long).to(device)
    task_id = DEF_MAPPING[source_type]
    
    with torch.no_grad():
        logits = model(inp, task_id=task_id)
        pred_idx = torch.argmax(logits, dim=1).item()
        pred_label = DEF_LABELS[pred_idx]
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        
    print(f'Source: {source_type}')
    print(f'Text  : "{text}"')
    print(f'Pred  : {pred_label} (Confidence: {probs[pred_idx]:.2%})\n')
    return pred_label

print('=== TEST INFERENSI DENGAN CONTOH RESMI DARI SATD_DETECTOR.PY ===\n')

# Test Comment
classify_satd('TODO: support multiple signers', source_type=DEF_COMMENT)
classify_satd('TODO: please add some javadoc', source_type=DEF_COMMENT)
classify_satd('TODO: lack of tests', source_type=DEF_COMMENT)

# Test Issue
classify_satd('I would like to remove this as its no longer needed.', source_type=DEF_ISSUE)
classify_satd('We need to update this documentation', source_type=DEF_ISSUE)
classify_satd('There are unimplemented requirements', source_type=DEF_ISSUE)
classify_satd('This is a good patch', source_type=DEF_ISSUE)

# Test Commit
classify_satd('fix bugs in SystemML - removed XXX', source_type=DEF_COMMIT)
classify_satd('fix typo in error message', source_type=DEF_COMMIT)

# Test Pull Request
classify_satd('nit: use local variable if possible', source_type=DEF_PULL)
classify_satd('Use the Python Postinstall implementation by default', source_type=DEF_PULL)